# Flight Delay Prediction — Feature Engineering

This notebook covers:
- **Section 1:** Rolling delay rate features (carrier, origin, destination — 30d/90d windows)
- **Section 2:** Target encoding for Origin and Dest airports (leakage-safe)

## Setup

In [ ]:
!pip install pyspark --quiet

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pyspark.sql.functions import col
import pyspark
import os
from pyspark.sql import SparkSession
from pyspark.sql import Window
from pyspark.sql import functions as F
import shutil, os, glob

In [ ]:
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

In [ ]:
spark = SparkSession.builder \
    .appName("ColabSpark") \
    .master("local[*]") \
    .config("spark.driver.memory", "8g") \
    .getOrCreate()

spark

In [ ]:
spark.conf.set("spark.sql.parquet.int96RebaseModeInRead", "CORRECTED")
spark.conf.set("spark.sql.legacy.parquet.nanosAsLong", "true")

## Section 1: Rolling Delay Rate Features

Computes backward-looking rolling delay rates (30d, 90d) per carrier, origin, and destination.
All windows are computed on the union of train/val/test to avoid re-scanning, but only past data is used for each row.

In [ ]:
DRIVE_PATH = '/content/drive/MyDrive/flights_data_pipeline'

# Read single pipeline output and split by year
df_all = spark.read.parquet(f'{DRIVE_PATH}/bts_with_weather_holiday.parquet')

df_train    = df_all.filter(df_all['Year'].isin([2018, 2019, 2020, 2021, 2022]))
df_validate = df_all.filter(df_all['Year'] == 2023)
df_test     = df_all.filter(df_all['Year'] == 2024)

print(f'Train:    {df_train.count():,} rows')
print(f'Validate: {df_validate.count():,} rows')
print(f'Test:     {df_test.count():,} rows')


In [ ]:
print(f"Train: {df_train.count():,} rows")
print(f"Val  : {df_validate.count():,} rows")
print(f"Test : {df_test.count():,} rows")

In [ ]:
df_train.dtypes

In [ ]:
# Save the training defaults

# ── Capture BEFORE transforming df_train ──────────────────────────────────────
# These are the "cold start" fill values derived purely from training data
global_delay_rate = df_train.agg(F.avg("ArrDel15")).collect()[0][0]
print(f"Global training delay rate (cold-start fill): {global_delay_rate:.4f}")

In [ ]:
# Tag both data frames and combine

# Tag each split so we can separate them later
df_train_tagged = df_train.withColumn("_split", F.lit("train"))
df_val_tagged   = df_validate.withColumn("_split",   F.lit("val"))
df_test_tagged  = df_test.withColumn("_split",  F.lit("test"))

# Union — all must have the same schema (ArrDel15 may be unknown but should exist)
df_all = (
    df_train_tagged
    .unionByName(df_val_tagged)
    .unionByName(df_test_tagged)
)

In [ ]:
# Build shared timestamp column

df_all = df_all.withColumn(
    "flight_date_unix",
    (F.col("FlightDate") / 1_000_000_000).cast("long")
)

df_all = df_all.withColumn(
    "crs_dep_unix",
    F.col("flight_date_unix")
    + (F.col("CRSDepTime") / 100).cast("int") * 3600
    + (F.col("CRSDepTime") % 100) * 60
)

In [ ]:
# Build all backward-looking windows

SECS_PER_DAY = 86_400
SECS_3H      = 3 * 3600

# Carrier delay rates
w_carrier_30d = (Window.partitionBy("Reporting_Airline")
                       .orderBy("flight_date_unix")
                       .rangeBetween(-30 * SECS_PER_DAY, -1))

w_carrier_90d = (Window.partitionBy("Reporting_Airline")
                       .orderBy("flight_date_unix")
                       .rangeBetween(-90 * SECS_PER_DAY, -1))

# Origin/Dest delay rates (target encoding)
w_origin_30d = (Window.partitionBy("Origin")
                      .orderBy("flight_date_unix")
                      .rangeBetween(-30 * SECS_PER_DAY, -1))

w_origin_90d = (Window.partitionBy("Origin")
                      .orderBy("flight_date_unix")
                      .rangeBetween(-90 * SECS_PER_DAY, -1))

w_dest_30d = (Window.partitionBy("Dest")
                    .orderBy("flight_date_unix")
                    .rangeBetween(-30 * SECS_PER_DAY, -1))

w_dest_90d = (Window.partitionBy("Dest")
                    .orderBy("flight_date_unix")
                    .rangeBetween(-90 * SECS_PER_DAY, -1))

# Congestion (3-hour window over scheduled departure timestamp)
w_congestion = (Window.partitionBy("Origin")
                      .orderBy("crs_dep_unix")
                      .rangeBetween(-SECS_3H, -1))


In [ ]:
# Compute all rolling features

df_all = (
    df_all
    .withColumn("carrier_delay_rate_30d", F.avg("ArrDel15").over(w_carrier_30d))
    .withColumn("carrier_delay_rate_90d", F.avg("ArrDel15").over(w_carrier_90d))
    .withColumn("origin_delay_rate_30d",  F.avg("ArrDel15").over(w_origin_30d))
    .withColumn("origin_delay_rate_90d",  F.avg("ArrDel15").over(w_origin_90d))
    .withColumn("dest_delay_rate_30d",    F.avg("ArrDel15").over(w_dest_30d))
    .withColumn("dest_delay_rate_90d",    F.avg("ArrDel15").over(w_dest_90d))
    .withColumn("origin_departures_3h",   F.count("*").over(w_congestion))
)

In [ ]:
# Fill nulls with training-derived defaults

df_all = df_all.fillna({
    "carrier_delay_rate_30d": global_delay_rate,
    "carrier_delay_rate_90d": global_delay_rate,
    "origin_delay_rate_30d":  global_delay_rate,
    "origin_delay_rate_90d":  global_delay_rate,
    "dest_delay_rate_30d":    global_delay_rate,
    "dest_delay_rate_90d":    global_delay_rate,
    "origin_departures_3h":   0,
})

In [ ]:
# Split back apart and drop construction columns

DROP_COLS = ["flight_date_unix", "crs_dep_unix", "_split"]

df_train = df_all.filter(F.col("_split") == "train").drop(*DROP_COLS)
df_val   = df_all.filter(F.col("_split") == "val").drop(*DROP_COLS)
df_test  = df_all.filter(F.col("_split") == "test").drop(*DROP_COLS)

In [ ]:
print(f"Train: {df_train.count():,} rows")
print(f"Val  : {df_val.count():,} rows")
print(f"Test : {df_test.count():,} rows")

In [ ]:
# Intermediate save skipped — DataFrames kept in memory for Section 2

In [ ]:
# write_single_parquet defined in Section 2 save cell

In [ ]:
# Intermediate save skipped — df_train, df_val, df_test passed directly to Section 2

## Section 2: Target Encoding for Origin and Dest

Recomputes `origin_delay_rate` and `dest_delay_rate` using a leakage-safe approach:
- **Train:** expanding window (each row sees only past flights at that airport)
- **Val/Test:** full training period mean per airport

# Target Encoding for Origin and Dest

This notebook recomputes `origin_delay_rate` and `dest_delay_rate` using a leakage-safe approach:

- **Training rows:** expanding window — each row receives the mean delay rate of its airport using only flights that occurred **before** that date. No row sees its own label or any future data.
- **Validation/Test rows:** full training period mean per airport (2018–2022 only). Val/test rows never touch their own `ArrDel15`.
- **Cold start:** airports with no prior history (first appearance, or unseen in training) fall back to the global training mean.


**Input:** `flights_all_features/` (train, val, test parquets — already split temporally)
- Train: 2018–2022
- Val: 2023
- Test: 2024

**Output:** `flights_all_features_encoded/` (same structure, with corrected encoding columns)

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql import Window

spark.sparkContext.setLogLevel('WARN')
print('Spark version:', spark.version)


In [ ]:
# Paths
OUT_PATH = '/content/drive/MyDrive/flight_data'
os.makedirs(OUT_PATH, exist_ok=True)

# Alias Section 1 DataFrames to Section 2 variable names
train_df = df_train
val_df   = df_validate
test_df  = df_test

print('Train rows:', train_df.count())
print('Val rows:  ', val_df.count())
print('Test rows: ', test_df.count())


In [ ]:
# Confirm Origin, Dest, ArrDel15, and FlightDate are present
required = ["Origin", "Dest", "ArrDel15", "FlightDate"]
missing  = [c for c in required if c not in train_df.columns]
print("Missing required columns:", missing if missing else "None — all present")
print("Total columns:", len(train_df.columns))

In [ ]:
# Drop existing (naively computed) encoding columns so we can recompute them
cols_to_drop = ["origin_delay_rate", "dest_delay_rate"]
existing     = [c for c in cols_to_drop if c in train_df.columns]

train_df = train_df.drop(*existing)
val_df   = val_df.drop(*existing)
test_df  = test_df.drop(*existing)

print(f"Dropped {existing} from all splits")

In [ ]:
# Global mean delay rate from training data
# Used as fallback for cold-start airports (no prior history)
global_mean = train_df.agg(F.mean("ArrDel15")).collect()[0][0]
print(f"Global training delay rate: {global_mean:.4f} ({global_mean*100:.2f}%)")

In [ ]:
# OOF expanding window encoding for TRAINING rows
#
# For each flight at airport X on date D:
#   origin_delay_rate = mean(ArrDel15) for all flights at X with FlightDate < D
#
# rowsBetween(unboundedPreceding, -1) excludes the current row and all rows after it
# in the sort order — so each row only sees past flights at that airport.

window_origin = (
    Window.partitionBy("Origin")
    .orderBy("FlightDate")
    .rowsBetween(Window.unboundedPreceding, -1)
)

window_dest = (
    Window.partitionBy("Dest")
    .orderBy("FlightDate")
    .rowsBetween(Window.unboundedPreceding, -1)
)

train_df = train_df.withColumn(
    "origin_delay_rate",
    F.coalesce(F.mean("ArrDel15").over(window_origin), F.lit(global_mean))
)

train_df = train_df.withColumn(
    "dest_delay_rate",
    F.coalesce(F.mean("ArrDel15").over(window_dest), F.lit(global_mean))
)

print("OOF encoding applied to training data")

In [ ]:
# Verify training encoding — spot check a few airports
train_df.select("FlightDate", "Origin", "ArrDel15", "origin_delay_rate") \
    .orderBy("Origin", "FlightDate") \
    .show(10, truncate=False)

In [ ]:
# Full training mean per airport — used for val and test lookup
# This is computed AFTER the OOF step so it reflects the final training distribution

origin_encoding = (
    train_df.groupBy("Origin")
    .agg(F.mean("ArrDel15").alias("origin_delay_rate"))
)

dest_encoding = (
    train_df.groupBy("Dest")
    .agg(F.mean("ArrDel15").alias("dest_delay_rate"))
)

print("Training encoding tables computed")
print("Unique origin airports:", origin_encoding.count())
print("Unique dest airports:  ", dest_encoding.count())

In [ ]:
# Apply training encoding to validation set
# Val rows (2023) receive the full 2018-2022 mean — they never touch their own ArrDel15

val_df = val_df.join(origin_encoding, on="Origin", how="left")
val_df = val_df.join(dest_encoding,   on="Dest",   how="left")

# Cold start: airports in val not seen in training get global mean
val_df = val_df.fillna({"origin_delay_rate": global_mean, "dest_delay_rate": global_mean})

print("Val encoding applied")
val_df.select("FlightDate", "Origin", "origin_delay_rate", "Dest", "dest_delay_rate").show(5)

In [ ]:
# Apply training encoding to test set
# Test rows (2024) receive the full 2018-2022 mean — same logic as val

test_df = test_df.join(origin_encoding, on="Origin", how="left")
test_df = test_df.join(dest_encoding,   on="Dest",   how="left")

# Cold start: airports in test not seen in training get global mean
test_df = test_df.fillna({"origin_delay_rate": global_mean, "dest_delay_rate": global_mean})

print("Test encoding applied")
test_df.select("FlightDate", "Origin", "origin_delay_rate", "Dest", "dest_delay_rate").show(5)

In [ ]:
# Sanity check — confirm no nulls in encoding columns across all splits
for name, df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    null_origin = df.filter(F.col("origin_delay_rate").isNull()).count()
    null_dest   = df.filter(F.col("dest_delay_rate").isNull()).count()
    print(f"{name} — null origin_delay_rate: {null_origin}, null dest_delay_rate: {null_dest}")

In [ ]:
# Sanity check — encoding range should be between 0 and 1
for name, df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    df.agg(
        F.min("origin_delay_rate").alias("origin_min"),
        F.max("origin_delay_rate").alias("origin_max"),
        F.mean("origin_delay_rate").alias("origin_mean"),
        F.min("dest_delay_rate").alias("dest_min"),
        F.max("dest_delay_rate").alias("dest_max"),
        F.mean("dest_delay_rate").alias("dest_mean"),
    ).show()
    print(f"^ {name}")

In [ ]:
import os, glob, shutil

def write_single_parquet(df, output_dir, filename):
    tmp_dir    = os.path.join(output_dir, f'_tmp_{filename}')
    final_path = os.path.join(output_dir, filename)
    (df.coalesce(1)
       .write
       .mode('overwrite')
       .option('compression', 'snappy')
       .parquet(tmp_dir))
    part_file = glob.glob(os.path.join(tmp_dir, 'part-*.parquet'))[0]
    if os.path.exists(final_path):
        os.remove(final_path)
    shutil.move(part_file, final_path)
    shutil.rmtree(tmp_dir)
    print(f'Saved {filename}: {df.count():,} rows → {final_path}')

os.makedirs(OUT_PATH, exist_ok=True)
write_single_parquet(train_df, OUT_PATH, 'train.parquet')
write_single_parquet(val_df,   OUT_PATH, 'val.parquet')
write_single_parquet(test_df,  OUT_PATH, 'test.parquet')
